In [1]:
# 1) Check GPU
!nvidia-smi

Wed Jun 10 13:34:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   34C    P0            100W /  600W |       0MiB /  97887MiB |    100%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
# 2) Install dependencies
!pip -q install -U uv

# Basic Python dependencies.
!uv pip install --system -U openai tqdm requests jsonschema psutil

# Install recent vLLM nightly for CUDA 13.0 / Blackwell.
# If this fails in your environment, use the auto backend line below instead.
!uv pip install --system -U vllm --torch-backend=cu130 --extra-index-url https://wheels.vllm.ai/nightly/cu130

# Fallback only if the cu130 line fails:
# !uv pip install --system -U vllm --torch-backend=auto --extra-index-url https://wheels.vllm.ai/nightly

# Version check
import torch
import vllm
import sys

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print("vLLM:", vllm.__version__)

Using Python 3.12.13 environment at: /usr
Resolved 25 packages in 80ms
Checked 25 packages in 0.28ms
Using Python 3.12.13 environment at: /usr
Resolved 190 packages in 9.96s
Prepared 1 package in 10.78s
Uninstalled 1 package in 38ms
Installed 1 package in 33ms
 - vllm==0.22.1rc1.dev350+g9ad08c4d1
 + vllm==0.22.1rc1.dev351+g9dfc313bd
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Torch: 2.11.0+cu130
Torch CUDA: 13.0
CUDA available: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
vLLM: 0.22.1rc1.dev351+g9dfc313bd


In [3]:
# 3) Mount Google Drive and prepare paths
from google.colab import drive
from pathlib import Path
import shutil
import json
import os

drive.mount("/content/drive")

GDRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/final_project/idea_1")
GDRIVE_INPUT_PATH = GDRIVE_PROJECT_DIR / "2wikimultihopqa_docs_chunks.json"

LOCAL_WORK_DIR = Path("/content/2wikimultihopqa_kg_test")
LOCAL_WORK_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_INPUT_PATH = LOCAL_WORK_DIR / "2wikimultihopqa_docs_chunks.json"

KG_DIR = GDRIVE_PROJECT_DIR / "kg" / "2wikimultihopqa"
KG_DIR.mkdir(parents=True, exist_ok=True)

assert GDRIVE_INPUT_PATH.exists(), f"Input file not found: {GDRIVE_INPUT_PATH}"

# Copy to local disk for faster reading.
shutil.copy2(GDRIVE_INPUT_PATH, LOCAL_INPUT_PATH)

print("Input:", LOCAL_INPUT_PATH)
print("Output folder:", KG_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Input: /content/2wikimultihopqa_kg_test/2wikimultihopqa_docs_chunks.json
Output folder: /content/drive/MyDrive/final_project/idea_1/kg/2wikimultihopqa


In [4]:
# 4) Start vLLM server - optimized throughput version
import subprocess
import time
import requests
from pathlib import Path
import os
import shlex
import psutil

MODEL_NAME = "Qwen/Qwen3.5-27B"
PORT = 8000
BASE_URL = f"http://localhost:{PORT}/v1"

# Total context length = prompt tokens + output tokens.
# 12288 is enough for ~1k-2k prompt + adaptive output up to 8192.
MAX_MODEL_LEN = 12288

# Keep memory margin on RTX PRO 6000 Blackwell 96GB.
GPU_MEMORY_UTILIZATION = 0.93

# Throughput settings.
MAX_NUM_SEQS = 8
MAX_NUM_BATCHED_TOKENS = 16384

SERVER_LOG_PATH = Path("/content/vllm_server.log")
SERVER_PID_PATH = Path("/content/vllm_server.pid")

def kill_process_tree(pid):
    # Kill a process and its children.
    try:
        parent = psutil.Process(int(pid))
        for child in parent.children(recursive=True):
            try:
                child.kill()
            except Exception:
                pass
        parent.kill()
        parent.wait(timeout=10)
        print("Killed old process tree:", pid)
    except Exception:
        pass

# Stop old PID from previous run.
if SERVER_PID_PATH.exists():
    old_pid = SERVER_PID_PATH.read_text().strip()
    if old_pid:
        kill_process_tree(old_pid)

# Kill any leftover vLLM server process.
for p in psutil.process_iter(["pid", "name", "cmdline"]):
    try:
        cmdline = " ".join(p.info.get("cmdline") or [])
        if "vllm" in cmdline and "serve" in cmdline:
            kill_process_tree(p.info["pid"])
            print("Killed leftover vLLM process:", p.info["pid"])
    except Exception:
        pass

time.sleep(3)

cmd = [
    "vllm", "serve", MODEL_NAME,

    "--host", "0.0.0.0",
    "--port", str(PORT),

    "--max-model-len", str(MAX_MODEL_LEN),
    "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION),

    # Text-only mode.
    "--language-model-only",

    # Qwen3 non-thinking mode at server level.
    "--reasoning-parser", "qwen3",
    "--default-chat-template-kwargs", '{"enable_thinking": false}',

    # Higher concurrency for throughput.
    "--max-num-seqs", str(MAX_NUM_SEQS),
    "--max-num-batched-tokens", str(MAX_NUM_BATCHED_TOKENS),

    # Prefix caching should help because the prompt prefix is mostly shared.
    "--enable-prefix-caching",

    # Use vLLM default generation config, not HF generation_config.json.
    "--generation-config", "vllm",

    # Explicit dtype for Blackwell.
    "--dtype", "bfloat16",

    # Safe for HF repos that need model code.
    "--trust-remote-code",
]

# Do NOT add --enforce-eager.
# Keeping torch.compile / CUDA graph path enabled should improve throughput after startup.

server_env = os.environ.copy()

# Keep this fix: FlashInfer sampler crashes in this Blackwell environment.
server_env["VLLM_USE_FLASHINFER_SAMPLER"] = "0"

# CUDA 13.0 / Blackwell runtime hints.
server_env["VLLM_MAIN_CUDA_VERSION"] = "13.0"
server_env["TORCH_CUDA_ARCH_LIST"] = "12.0"

print("Command:")
print(" ".join(shlex.quote(x) for x in cmd))

print("\nImportant environment variables:")
for k in ["VLLM_USE_FLASHINFER_SAMPLER", "VLLM_MAIN_CUDA_VERSION", "TORCH_CUDA_ARCH_LIST"]:
    print(f"{k}={server_env.get(k)}")

SERVER_LOG_PATH.write_text("", encoding="utf-8")
log_file = open(SERVER_LOG_PATH, "w", encoding="utf-8")

proc = subprocess.Popen(
    cmd,
    stdout=log_file,
    stderr=subprocess.STDOUT,
    text=True,
    env=server_env,
)

SERVER_PID_PATH.write_text(str(proc.pid))

print("\nStarted vLLM server.")
print("PID:", proc.pid)
print("Log:", SERVER_LOG_PATH)

Command:
vllm serve Qwen/Qwen3.5-27B --host 0.0.0.0 --port 8000 --max-model-len 12288 --gpu-memory-utilization 0.93 --language-model-only --reasoning-parser qwen3 --default-chat-template-kwargs '{"enable_thinking": false}' --max-num-seqs 8 --max-num-batched-tokens 16384 --enable-prefix-caching --generation-config vllm --dtype bfloat16 --trust-remote-code

Important environment variables:
VLLM_USE_FLASHINFER_SAMPLER=0
VLLM_MAIN_CUDA_VERSION=13.0
TORCH_CUDA_ARCH_LIST=12.0

Started vLLM server.
PID: 6330
Log: /content/vllm_server.log


In [5]:
# 5) Wait for vLLM server with compact logging
import time
import requests
from pathlib import Path

def tail_log(path, n=80):
    path = Path(path)
    if not path.exists():
        return ""
    lines = path.read_text(errors="ignore").splitlines()
    return "\n".join(lines[-n:])

ready = False

MAX_WAIT_SEC = 1800
SLEEP_SEC = 5
PRINT_EVERY_SEC = 60

start = time.perf_counter()
last_print = -PRINT_EVERY_SEC

for step in range(MAX_WAIT_SEC // SLEEP_SEC):
    elapsed = int(time.perf_counter() - start)

    return_code = proc.poll()
    if return_code is not None:
        print(f"vLLM process exited. Return code: {return_code}")
        print("\n=== Last vLLM log lines ===")
        print(tail_log(SERVER_LOG_PATH, n=160))
        raise RuntimeError("vLLM server crashed or exited during startup.")

    try:
        h = requests.get(f"http://localhost:{PORT}/health", timeout=5)
        if h.status_code == 200:
            m = requests.get(f"{BASE_URL}/models", timeout=10)
            if m.status_code == 200:
                ready = True
                model_info = m.json()["data"][0]
                print("vLLM server is ready.")
                print("Model:", model_info["id"])
                print("Max model len:", model_info.get("max_model_len"))
                break
    except Exception:
        pass

    if elapsed - last_print >= PRINT_EVERY_SEC:
        last_print = elapsed
        print(f"Waiting... {elapsed}s")
        recent = tail_log(SERVER_LOG_PATH, n=12)
        if recent.strip():
            print(recent)
        print("-" * 80)

    time.sleep(SLEEP_SEC)

if not ready:
    print("\n=== Last vLLM log lines ===")
    print(tail_log(SERVER_LOG_PATH, n=160))
    raise RuntimeError("vLLM server did not become ready before timeout.")

Waiting... 0s
--------------------------------------------------------------------------------
Waiting... 60s
Loading safetensors checkpoint shards: 100% Completed | 11/11 [00:05<00:00,  1.94it/s]
(EngineCore pid=6609) 
(EngineCore pid=6609) INFO 06-10 13:35:26 [default_loader.py:397] Loading weights took 5.71 seconds
(EngineCore pid=6609) INFO 06-10 13:35:26 [gpu_model_runner.py:5182] Model loading took 50.22 GiB memory and 7.577976 seconds
(EngineCore pid=6609) INFO 06-10 13:35:26 [interface.py:670] Setting attention block size to 784 tokens to ensure that attention page size is >= mamba page size.
(EngineCore pid=6609) INFO 06-10 13:35:26 [interface.py:694] Padding mamba page size by 0.13% to ensure that mamba page size and attention page size are exactly equal.
(EngineCore pid=6609) INFO 06-10 13:35:32 [backends.py:1089] Using cache directory: /root/.cache/vllm/torch_compile_cache/d55973fcbe/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=6609) INFO 06-10 13:35:32 [backe

In [6]:
# 6) Load all chunks and validate chunk ids
import json
import re
from pathlib import Path

with open(LOCAL_INPUT_PATH, "r", encoding="utf-8") as f:
    chunks = json.load(f)

assert isinstance(chunks, list), "The input JSON must be a list of chunks."

def expected_chunk_id(input_index):
    return f"2wikimultihopqa_chunk_{input_index + 1:08d}"

chunk_id_mismatches = []

for i, chunk in enumerate(chunks):
    got = chunk.get("Chunk_id")
    expected = expected_chunk_id(i)

    if got != expected:
        chunk_id_mismatches.append({
            "input_index": i,
            "expected_chunk_id": expected,
            "got_chunk_id": got,
        })

print("Total chunks:", len(chunks))
print("First chunk:", chunks[0]["Chunk_id"])
print("Last chunk:", chunks[-1]["Chunk_id"])

if chunk_id_mismatches:
    print("Chunk id mismatches:", len(chunk_id_mismatches))
    print(json.dumps(chunk_id_mismatches[:10], ensure_ascii=False, indent=2))
    raise RuntimeError("Chunk_id order does not match input_index order.")

print("Chunk id order is valid.")
print(json.dumps(chunks[0], ensure_ascii=False, indent=2)[:2000])

Total chunks: 12685
First chunk: 2wikimultihopqa_chunk_00000001
Last chunk: 2wikimultihopqa_chunk_00012685
Chunk id order is valid.
{
  "Chunk_id": "2wikimultihopqa_chunk_00000001",
  "Title": "Calloway County High School",
  "Paragraph_id": [
    1,
    2,
    3,
    4
  ],
  "Text": "Calloway County High School is a public high school located in Murray, Kentucky. The school was formed from the consolidation of six high schools from across the county: Hazel High School, Lynn Grove High School, Kirksey High School, Almo High School, New Concord High School, and Faxon High School.\nOrganizations: Clubs/Organizations\nState champions: Wrestling: David Woods 195 lbs (2017) Bass Fishing: Bracken Robertson & Dillon Starks (2013) Boys Cross Country: 1984 (2A) Fast Pitch Softball: 2004 Girls Golf: 2012 (Individual, Anna Hack)\nW. Earl Brown, actor: Pookie Jones, 1989 KHSAA Mr. Football winner*",
  "Token_count": 153
}


In [7]:
# 7) Define schema and prompt
import json

KG_SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "properties": {
        "entities": {
            "type": "array",
            "maxItems": 30,
            "items": {
                "type": "string",
                "minLength": 1
            }
        },
        "relations": {
            "type": "array",
            "maxItems": 35,
            "items": {
                "type": "object",
                "additionalProperties": False,
                "properties": {
                    "head": {
                        "type": "string",
                        "minLength": 1
                    },
                    "relation": {
                        "type": "string",
                        "minLength": 1
                    },
                    "tail": {
                        "type": "string",
                        "minLength": 1
                    }
                },
                "required": ["head", "relation", "tail"]
            }
        },
        "facts": {
            "type": "array",
            "maxItems": 30,
            "items": {
                "type": "object",
                "additionalProperties": False,
                "properties": {
                    "entity": {
                        "type": "string",
                        "minLength": 1
                    },
                    "info": {
                        "type": "string",
                        "minLength": 1
                    }
                },
                "required": ["entity", "info"]
            }
        }
    },
    "required": ["entities", "relations", "facts"]
}

EXAMPLE_OUTPUT = {
    "entities": [
        "Marie Curie",
        "radioactivity",
        "Pierre Curie",
        "Curie Institute",
        "Paris",
        "1920"
    ],
    "relations": [
        {
            "head": "Marie Curie",
            "relation": "Marie Curie conducted pioneering research on radioactivity.",
            "tail": "radioactivity"
        },
        {
            "head": "Marie Curie",
            "relation": "Marie Curie was married to Pierre Curie.",
            "tail": "Pierre Curie"
        },
        {
            "head": "Curie Institute",
            "relation": "The Curie Institute was founded in Paris.",
            "tail": "Paris"
        },
        {
            "head": "Curie Institute",
            "relation": "The Curie Institute was founded in 1920.",
            "tail": "1920"
        }
    ],
    "facts": [
        {
            "entity": "Marie Curie",
            "info": "Marie Curie was a Polish and naturalized-French physicist and chemist who researched radioactivity."
        },
        {
            "entity": "Pierre Curie",
            "info": "Pierre Curie was married to Marie Curie."
        },
        {
            "entity": "Curie Institute",
            "info": "The Curie Institute in Paris was founded in 1920."
        }
    ]
}

def build_prompt(chunk):
    chunk_id = chunk["Chunk_id"]
    title = chunk["Title"]
    paragraph_ids = json.dumps(chunk["Paragraph_id"], ensure_ascii=False)
    chunk_text = chunk["Text"]

    return f"""You are an expert information extraction system designed to build a highly accurate retrieval knowledge graph.

Your task is to extract entities, relations between entities, and specific facts about entities from a given Wikipedia chunk.

Extract only facts explicitly supported by the chunk.
Do not use external knowledge.
Do not infer facts that are not clearly stated.
Prefer precision over recall.
Return JSON only.

Extract:

1. entities:
Important specific entities useful for multi-hop retrieval.
Include specific people, organizations, locations, works, events, awards, dates/years, and key concepts.
Do not extract generic adjectives or common nouns as standalone entities.
Extract at most 30 entities.

2. relations:
A relation is a connection between two extracted entities that is explicitly stated or directly supported by the chunk.
Each relation must have:
- head: one entity copied exactly from entities
- tail: one entity copied exactly from entities
- relation: a short, simple natural-language sentence explaining the connection between head and tail
Extract at most 35 relations.

3. facts:
A fact is a short, simple natural-language sentence describing what specific information this exact chunk provides about one extracted entity.
Each fact must have:
- entity: one entity copied exactly from entities
- info: a short sentence about that entity based only on this chunk
Extract at most 30 facts.

Rules:
- Every head and tail in relations must be copied exactly from the entities array.
- Every entity in facts must be copied exactly from the entities array.
- Do not create duplicate entities, duplicate relations, or duplicate facts.
- Resolve pronouns to actual entity names only when unambiguous.
- Generic words may appear inside relation and info sentences, but not as standalone entities.
- The output must be valid JSON strictly matching the required schema.

Example:

Input:
chunk_id:
ex_001

title:
Marie Curie

paragraph_ids:
[1]

text:
Marie Curie was a Polish and naturalized-French physicist and chemist who conducted pioneering research on radioactivity. She was married to Pierre Curie. The Curie Institute in Paris was founded in 1920.

Output:
{json.dumps(EXAMPLE_OUTPUT, ensure_ascii=False, indent=2)}

Now extract from this chunk.

Input:
chunk_id:
{chunk_id}

title:
{title}

paragraph_ids:
{paragraph_ids}

text:
{chunk_text}

Output:
"""

In [8]:
# 8) Define client and adaptive extraction helpers
import json
import time
from openai import OpenAI

client = OpenAI(
    api_key="EMPTY",
    base_url=BASE_URL,
    timeout=3600,
)

# Adaptive output budget.
OUTPUT_TOKEN_STEPS = [4096, 8192]
MAX_OUTPUT_TOKENS = max(OUTPUT_TOKEN_STEPS)

# Qwen3.5 non-thinking general-task parameters.
TEMPERATURE = 0.7
TOP_P = 0.8
TOP_K = 20
MIN_P = 0.0

# For information extraction, 0.0 is safer than 1.5.
PRESENCE_PENALTY = 0.0
REPETITION_PENALTY = 1.0

SAVE_RAW_RESPONSE_ON_ERROR = True

def make_messages(chunk):
    return [
        {
            "role": "system",
            "content": "You extract knowledge graph data from Wikipedia chunks. Return valid JSON only."
        },
        {
            "role": "user",
            "content": build_prompt(chunk)
        }
    ]

def usage_to_dict(usage):
    if usage is None:
        return None

    return {
        "prompt_tokens": getattr(usage, "prompt_tokens", None),
        "completion_tokens": getattr(usage, "completion_tokens", None),
        "total_tokens": getattr(usage, "total_tokens", None),
    }

def call_llm_once(chunk, max_tokens):
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=make_messages(chunk),
        max_tokens=max_tokens,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        presence_penalty=PRESENCE_PENALTY,
        seed=42,
        extra_body={
            "top_k": TOP_K,
            "min_p": MIN_P,
            "repetition_penalty": REPETITION_PENALTY,
            "chat_template_kwargs": {
                "enable_thinking": False
            },
            "structured_outputs": {
                "json": KG_SCHEMA
            }
        },
    )

    choice = response.choices[0]

    return {
        "content": choice.message.content,
        "finish_reason": choice.finish_reason,
        "usage": usage_to_dict(response.usage),
        "max_tokens": max_tokens,
    }

def extract_one_raw_adaptive(chunk, max_retries_per_budget=1):
    # Store exactly the model JSON fields after JSON parsing.
    start = time.perf_counter()
    last_error = None
    raw = None
    last_finish_reason = None
    last_usage = None
    last_max_tokens = None
    attempts = []

    for max_tokens in OUTPUT_TOKEN_STEPS:
        for retry_id in range(max_retries_per_budget + 1):
            try:
                response_data = call_llm_once(chunk, max_tokens=max_tokens)

                raw = response_data["content"]
                finish_reason = response_data["finish_reason"]
                usage = response_data["usage"]

                last_finish_reason = finish_reason
                last_usage = usage
                last_max_tokens = max_tokens

                attempts.append({
                    "max_tokens": max_tokens,
                    "retry_id": retry_id,
                    "finish_reason": finish_reason,
                    "usage": usage,
                })

                # If the model hit the token budget, retry with a larger budget.
                if finish_reason == "length":
                    last_error = f"finish_reason=length at max_tokens={max_tokens}"
                    break

                # Parse only to make final saved file valid JSON.
                model_output = json.loads(raw)

                return {
                    "chunk_id": chunk["Chunk_id"],
                    "title": chunk["Title"],
                    "paragraph_id": chunk["Paragraph_id"],
                    "token_count": chunk.get("Token_count"),

                    # Model output copied as-is after JSON parsing.
                    "entities": model_output.get("entities"),
                    "relations": model_output.get("relations"),
                    "facts": model_output.get("facts"),

                    "error": None,
                    "latency_sec": round(time.perf_counter() - start, 3),
                    "finish_reason": finish_reason,
                    "max_tokens_used": max_tokens,
                    "usage": usage,
                    "attempts": attempts,
                }

            except Exception as e:
                last_error = repr(e)
                time.sleep(1.0 * (retry_id + 1))

        # Continue to the next larger max_tokens budget.

    failed_item = {
        "chunk_id": chunk["Chunk_id"],
        "title": chunk["Title"],
        "paragraph_id": chunk["Paragraph_id"],
        "token_count": chunk.get("Token_count"),
        "entities": None,
        "relations": None,
        "facts": None,
        "error": last_error,
        "latency_sec": round(time.perf_counter() - start, 3),
        "finish_reason": last_finish_reason,
        "max_tokens_used": last_max_tokens,
        "usage": last_usage,
        "attempts": attempts,
    }

    if SAVE_RAW_RESPONSE_ON_ERROR:
        failed_item["raw_response"] = raw

    return failed_item

In [9]:
# 9) Audit existing KG files and find truly missing or failed indices
import json
import os
import re
from pathlib import Path
from collections import defaultdict, Counter
from datetime import datetime, timezone

DATASET_NAME = "2wikimultihopqa"

FINAL_OUTPUT_RE = re.compile(
    r"^2wikimultihopqa_kg_extractions_(?:first\d+_)?chunks_\d{8}_to_\d{8}\.json$"
)

AUDIT_REPORT_PATH = KG_DIR / "2wikimultihopqa_kg_audit_report_before_repair.json"
REPAIR_INDEX_PATH = KG_DIR / "2wikimultihopqa_missing_or_failed_indices.json"

REPROCESS_ERROR_ITEMS = True

def atomic_json_dump(obj, path):
    # Write JSON safely, then atomically replace the target file.
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_name(path.name + ".tmp")

    with open(tmp_path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)
        f.flush()
        os.fsync(f.fileno())

    os.replace(tmp_path, path)

def summarize_ranges(values, max_ranges=30):
    # Convert sorted integers into compact ranges.
    values = sorted(set(values))

    if not values:
        return []

    ranges = []
    start = prev = values[0]

    for x in values[1:]:
        if x == prev + 1:
            prev = x
        else:
            ranges.append((start, prev))
            start = prev = x

    ranges.append((start, prev))

    out = [
        str(a) if a == b else f"{a}-{b}"
        for a, b in ranges[:max_ranges]
    ]

    if len(ranges) > max_ranges:
        out.append(f"... plus {len(ranges) - max_ranges} more ranges")

    return out

def input_index_from_item(item):
    # Prefer explicit input_index, otherwise recover it from chunk_id.
    idx = item.get("input_index")

    if isinstance(idx, int):
        return idx

    chunk_id = item.get("chunk_id")

    if isinstance(chunk_id, str):
        m = re.search(r"2wikimultihopqa_chunk_(\d{8})$", chunk_id)
        if m:
            return int(m.group(1)) - 1

    return None

def parse_range_from_filename(path):
    # Parse declared chunk-number range from the filename.
    m = re.search(r"chunks_(\d{8})_to_(\d{8})\.json$", path.name)

    if not m:
        return None

    first_chunk_number = int(m.group(1))
    last_chunk_number = int(m.group(2))

    return {
        "first_chunk_number": first_chunk_number,
        "last_chunk_number": last_chunk_number,
        "start_index": first_chunk_number - 1,
        "end_index_exclusive": last_chunk_number,
    }

def is_usable_success(item, idx):
    # HotpotQA-style success check.
    # This intentionally does NOT enforce relation head/tail membership in entities.
    if not isinstance(item, dict):
        return False

    if not isinstance(idx, int) or idx < 0 or idx >= len(chunks):
        return False

    if item.get("chunk_id") != expected_chunk_id(idx):
        return False

    if item.get("error") is not None:
        return False

    if item.get("finish_reason") == "length":
        return False

    if not isinstance(item.get("entities"), list):
        return False

    if not isinstance(item.get("relations"), list):
        return False

    if not isinstance(item.get("facts"), list):
        return False

    return True

def failure_reasons_hotpot_style(item, idx):
    # Report only real structural/run failures.
    reasons = []

    if not isinstance(item, dict):
        return ["item_not_dict"]

    if not isinstance(idx, int) or idx < 0 or idx >= len(chunks):
        return ["invalid_input_index"]

    if item.get("chunk_id") != expected_chunk_id(idx):
        reasons.append("chunk_id_mismatch")

    if item.get("error") is not None:
        reasons.append("error_not_none")

    if item.get("finish_reason") == "length":
        reasons.append("finish_reason_length")

    if not isinstance(item.get("entities"), list):
        reasons.append("entities_not_list")

    if not isinstance(item.get("relations"), list):
        reasons.append("relations_not_list")

    if not isinstance(item.get("facts"), list):
        reasons.append("facts_not_list")

    return sorted(set(reasons))

final_paths = sorted(
    p for p in KG_DIR.glob("2wikimultihopqa_kg_extractions_*.json")
    if FINAL_OUTPUT_RE.match(p.name)
)

assert final_paths, f"No final extraction files found in: {KG_DIR}"

records_by_index = defaultdict(list)
file_summaries = []
invalid_records = []

for path in final_paths:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, list):
        invalid_records.append({
            "file": str(path),
            "reason": "File content is not a list.",
        })
        continue

    parsed_range = parse_range_from_filename(path)
    indices_in_file = []
    errors_in_file = 0
    unusable_in_file = 0

    for local_pos, item in enumerate(data):
        if not isinstance(item, dict):
            invalid_records.append({
                "file": str(path),
                "local_pos": local_pos,
                "reason": "Item is not a dict.",
            })
            continue

        idx = input_index_from_item(item)

        if not isinstance(idx, int):
            invalid_records.append({
                "file": str(path),
                "local_pos": local_pos,
                "chunk_id": item.get("chunk_id"),
                "reason": "Missing or invalid input_index and cannot recover from chunk_id.",
            })
            continue

        if idx < 0 or idx >= len(chunks):
            invalid_records.append({
                "file": str(path),
                "local_pos": local_pos,
                "input_index": idx,
                "chunk_id": item.get("chunk_id"),
                "reason": "input_index out of range.",
            })
            continue

        usable_success = is_usable_success(item, idx)
        failure_reasons = failure_reasons_hotpot_style(item, idx)

        if item.get("error") is not None:
            errors_in_file += 1

        if not usable_success:
            unusable_in_file += 1

        indices_in_file.append(idx)

        records_by_index[idx].append({
            "file": str(path),
            "filename": path.name,
            "local_pos": local_pos,
            "chunk_id": item.get("chunk_id"),
            "error": item.get("error"),
            "usable_success": usable_success,
            "failure_reasons": failure_reasons,
        })

    file_summaries.append({
        "file": str(path),
        "filename": path.name,
        "declared_range": parsed_range,
        "num_items": len(data),
        "min_input_index": min(indices_in_file) if indices_in_file else None,
        "max_input_index": max(indices_in_file) if indices_in_file else None,
        "num_unique_input_indices": len(set(indices_in_file)),
        "num_errors": errors_in_file,
        "num_unusable_records": unusable_in_file,
    })

all_indices = set(range(len(chunks)))
covered_indices = set(records_by_index.keys())
missing_input_indices = sorted(all_indices - covered_indices)

duplicate_input_indices = sorted(
    idx for idx, rows in records_by_index.items()
    if len(rows) > 1
)

error_input_indices = sorted(
    idx for idx, rows in records_by_index.items()
    if any(row.get("error") is not None for row in rows)
)

unusable_input_indices = sorted(
    idx for idx, rows in records_by_index.items()
    if not any(row["usable_success"] for row in rows)
)

repair_reasons = defaultdict(set)

for idx in missing_input_indices:
    repair_reasons[idx].add("missing")

for idx in unusable_input_indices:
    rows = records_by_index.get(idx, [])

    if not rows:
        continue

    all_reasons = set()
    for row in rows:
        all_reasons.update(row["failure_reasons"])

    for reason in sorted(all_reasons):
        repair_reasons[idx].add(reason)

repair_input_indices = sorted(repair_reasons.keys())

repair_items = [
    {
        "input_index": idx,
        "chunk_number": idx + 1,
        "chunk_id": chunks[idx]["Chunk_id"],
        "title": chunks[idx].get("Title"),
        "reasons": sorted(repair_reasons[idx]),
    }
    for idx in repair_input_indices
]

audit_report = {
    "dataset": DATASET_NAME,
    "input_path": str(GDRIVE_INPUT_PATH),
    "kg_dir": str(KG_DIR),
    "total_chunks": len(chunks),
    "num_final_files": len(final_paths),
    "final_files": file_summaries,
    "num_covered_input_indices": len(covered_indices),
    "num_missing_input_indices": len(missing_input_indices),
    "num_duplicate_input_indices": len(duplicate_input_indices),
    "num_error_input_indices": len(error_input_indices),
    "num_unusable_input_indices": len(unusable_input_indices),
    "num_repair_input_indices": len(repair_input_indices),
    "missing_input_index_ranges_0_based": summarize_ranges(missing_input_indices),
    "missing_chunk_number_ranges_1_based": summarize_ranges([i + 1 for i in missing_input_indices]),
    "error_input_index_ranges_0_based": summarize_ranges(error_input_indices),
    "error_chunk_number_ranges_1_based": summarize_ranges([i + 1 for i in error_input_indices]),
    "unusable_input_index_ranges_0_based": summarize_ranges(unusable_input_indices),
    "unusable_chunk_number_ranges_1_based": summarize_ranges([i + 1 for i in unusable_input_indices]),
    "repair_input_index_ranges_0_based": summarize_ranges(repair_input_indices),
    "repair_chunk_number_ranges_1_based": summarize_ranges([i + 1 for i in repair_input_indices]),
    "duplicate_input_indices": duplicate_input_indices,
    "invalid_records": invalid_records,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
}

atomic_json_dump(audit_report, AUDIT_REPORT_PATH)
atomic_json_dump(repair_items, REPAIR_INDEX_PATH)

print("Final files found:", len(final_paths))
for s in file_summaries:
    print(
        s["filename"],
        "| items:", s["num_items"],
        "| unique:", s["num_unique_input_indices"],
        "| errors:", s["num_errors"],
        "| unusable:", s["num_unusable_records"],
        "| min/max input_index:", s["min_input_index"], s["max_input_index"],
    )

print("\nCovered input indices:", len(covered_indices), "/", len(chunks))
print("Missing input indices:", len(missing_input_indices))
print("Missing chunk numbers:", summarize_ranges([i + 1 for i in missing_input_indices]))
print("Duplicate input indices:", len(duplicate_input_indices))
print("Error input indices:", len(error_input_indices))
print("Unusable input indices:", len(unusable_input_indices))
print("Repair input indices:", len(repair_input_indices))
print("Repair chunk numbers:", summarize_ranges([i + 1 for i in repair_input_indices]))

print("\nSaved audit report:", AUDIT_REPORT_PATH)
print("Saved repair index list:", REPAIR_INDEX_PATH)

Final files found: 3
2wikimultihopqa_kg_extractions_chunks_00005001_to_00010000.json | items: 5000 | unique: 5000 | errors: 6 | unusable: 6 | min/max input_index: 5000 9999
2wikimultihopqa_kg_extractions_chunks_00010001_to_00012685.json | items: 2685 | unique: 2685 | errors: 3 | unusable: 3 | min/max input_index: 10000 12684
2wikimultihopqa_kg_extractions_first5000_chunks_00000001_to_00005000.json | items: 5000 | unique: 5000 | errors: 7 | unusable: 7 | min/max input_index: 0 4999

Covered input indices: 12685 / 12685
Missing input indices: 0
Missing chunk numbers: []
Duplicate input indices: 0
Error input indices: 16
Unusable input indices: 16
Repair input indices: 16
Repair chunk numbers: ['1583', '1609', '1672', '1785', '2350', '3804', '4796', '5626', '6046', '6349', '7688', '8736', '8919', '10634', '10637', '10779']

Saved audit report: /content/drive/MyDrive/final_project/idea_1/kg/2wikimultihopqa/2wikimultihopqa_kg_audit_report_before_repair.json
Saved repair index list: /content

In [10]:
# 10) Repair truly missing or failed chunks
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
import json
import os
import time
from datetime import datetime, timezone
from collections import Counter

REPAIR_OUTPUT_PATH = KG_DIR / "2wikimultihopqa_kg_extractions_repair_missing_failed.json"
REPAIR_META_PATH = KG_DIR / "2wikimultihopqa_kg_extractions_repair_missing_failed_meta.json"
REPAIR_PARTIAL_PATH = KG_DIR / "2wikimultihopqa_kg_extractions_repair_missing_failed.partial.json"

MAX_WORKERS = 8
SAVE_EVERY = 5
RETRY_FAILED_REPAIR_ITEMS = True

# Use larger budgets than the original run, but keep them within the server max model length.
REPAIR_OUTPUT_TOKEN_STEPS = [4096, 8192, 10000]

print("Repair items to process:", len(repair_input_indices))
print("Repair chunk numbers:", summarize_ranges([i + 1 for i in repair_input_indices]))

def call_llm_repair_once(chunk, max_tokens):
    # Same original prompt and schema, but safer deterministic decoding.
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=make_messages(chunk),
        max_tokens=max_tokens,
        temperature=0.0,
        top_p=1.0,
        presence_penalty=0.0,
        seed=101,
        extra_body={
            "top_k": 20,
            "min_p": 0.0,
            "repetition_penalty": 1.12,
            "chat_template_kwargs": {
                "enable_thinking": False
            },
            "structured_outputs": {
                "json": KG_SCHEMA
            },
        },
    )

    choice = response.choices[0]

    return {
        "content": choice.message.content,
        "finish_reason": choice.finish_reason,
        "usage": usage_to_dict(response.usage),
        "max_tokens": max_tokens,
    }

def validate_repair_payload_hotpot_style(model_output):
    # HotpotQA-style validation: only ensure the three top-level arrays exist.
    if not isinstance(model_output, dict):
        raise ValueError("Model output is not a dict.")

    if not isinstance(model_output.get("entities"), list):
        raise ValueError("entities is not a list.")

    if not isinstance(model_output.get("relations"), list):
        raise ValueError("relations is not a list.")

    if not isinstance(model_output.get("facts"), list):
        raise ValueError("facts is not a list.")

def extract_one_repair(chunk, max_retries_per_budget=2):
    # Repair extraction using original prompt/schema and larger token budgets.
    start = time.perf_counter()
    last_error = None
    raw = None
    last_finish_reason = None
    last_usage = None
    last_max_tokens = None
    attempts = []

    for max_tokens in REPAIR_OUTPUT_TOKEN_STEPS:
        for retry_id in range(max_retries_per_budget + 1):
            try:
                response_data = call_llm_repair_once(chunk, max_tokens=max_tokens)

                raw = response_data["content"]
                finish_reason = response_data["finish_reason"]
                usage = response_data["usage"]

                last_finish_reason = finish_reason
                last_usage = usage
                last_max_tokens = max_tokens

                attempts.append({
                    "max_tokens": max_tokens,
                    "retry_id": retry_id,
                    "finish_reason": finish_reason,
                    "usage": usage,
                })

                if finish_reason == "length":
                    last_error = f"finish_reason=length at max_tokens={max_tokens}"
                    break

                model_output = json.loads(raw)
                validate_repair_payload_hotpot_style(model_output)

                return {
                    "chunk_id": chunk["Chunk_id"],
                    "title": chunk["Title"],
                    "paragraph_id": chunk["Paragraph_id"],
                    "token_count": chunk.get("Token_count"),
                    "entities": model_output.get("entities"),
                    "relations": model_output.get("relations"),
                    "facts": model_output.get("facts"),
                    "error": None,
                    "latency_sec": round(time.perf_counter() - start, 3),
                    "finish_reason": finish_reason,
                    "max_tokens_used": max_tokens,
                    "usage": usage,
                    "attempts": attempts,
                    "repair_run": True,
                    "repair_pass": "repair_missing_failed",
                }

            except Exception as e:
                last_error = repr(e)
                time.sleep(1.0 * (retry_id + 1))

    failed_item = {
        "chunk_id": chunk["Chunk_id"],
        "title": chunk["Title"],
        "paragraph_id": chunk["Paragraph_id"],
        "token_count": chunk.get("Token_count"),
        "entities": None,
        "relations": None,
        "facts": None,
        "error": last_error,
        "latency_sec": round(time.perf_counter() - start, 3),
        "finish_reason": last_finish_reason,
        "max_tokens_used": last_max_tokens,
        "usage": last_usage,
        "attempts": attempts,
        "repair_run": True,
        "repair_pass": "repair_missing_failed",
    }

    failed_item["raw_response"] = raw
    return failed_item

def load_json_list_if_exists(path):
    # Load a JSON list if path exists.
    if not path.exists():
        return []

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, list):
        raise ValueError(f"Expected JSON list: {path}")

    return data

def load_existing_repair_items():
    # Prefer final repair output, then repair checkpoint.
    for path in [REPAIR_OUTPUT_PATH, REPAIR_PARTIAL_PATH]:
        if path.exists():
            try:
                return load_json_list_if_exists(path)
            except Exception:
                pass

    return []

def save_repair_partial(results_by_index):
    # Save completed repair items only.
    partial_results = [
        results_by_index[idx]
        for idx in sorted(results_by_index)
    ]
    atomic_json_dump(partial_results, REPAIR_PARTIAL_PATH)

def make_repair_failed_item(idx, chunk, error):
    # Store unexpected failures without stopping the repair run.
    return {
        "chunk_id": chunk["Chunk_id"],
        "title": chunk["Title"],
        "paragraph_id": chunk["Paragraph_id"],
        "token_count": chunk.get("Token_count"),
        "entities": None,
        "relations": None,
        "facts": None,
        "error": error,
        "latency_sec": None,
        "finish_reason": None,
        "max_tokens_used": None,
        "usage": None,
        "attempts": [],
        "input_index": idx,
        "repair_run": True,
        "repair_pass": "repair_missing_failed",
    }

repair_index_to_local = {
    idx: local_pos
    for local_pos, idx in enumerate(repair_input_indices)
}

results_by_index = {}

existing_repair_items = load_existing_repair_items()

for item in existing_repair_items:
    idx = input_index_from_item(item)

    if idx not in repair_index_to_local:
        continue

    if RETRY_FAILED_REPAIR_ITEMS and not is_usable_success(item, idx):
        continue

    item["input_index"] = idx
    item["repair_local_index"] = repair_index_to_local[idx]
    results_by_index[idx] = item

pending_jobs = [
    (idx, chunks[idx])
    for idx in repair_input_indices
    if idx not in results_by_index
]

print("Already completed usable repair items:", len(results_by_index))
print("Pending repair items:", len(pending_jobs))

def run_one_repair(index_and_chunk):
    idx, chunk = index_and_chunk

    item = extract_one_repair(chunk, max_retries_per_budget=2)

    item["input_index"] = idx
    item["repair_local_index"] = repair_index_to_local[idx]
    item["repair_run"] = True
    item["repair_pass"] = "repair_missing_failed"

    return idx, item

overall_start = time.perf_counter()
completed_new = 0
executor = None
cancelled = False

if pending_jobs:
    try:
        executor = ThreadPoolExecutor(max_workers=MAX_WORKERS)

        futures = {
            executor.submit(run_one_repair, job): job
            for job in pending_jobs
        }

        for future in tqdm(
            as_completed(futures),
            total=len(futures),
            desc="Repairing missing or failed KG items"
        ):
            idx, chunk = futures[future]

            try:
                _, item = future.result()
            except Exception as e:
                item = make_repair_failed_item(idx, chunk, repr(e))

            results_by_index[idx] = item
            completed_new += 1

            if completed_new % SAVE_EVERY == 0:
                save_repair_partial(results_by_index)

    except KeyboardInterrupt:
        cancelled = True
        save_repair_partial(results_by_index)

        if executor is not None:
            executor.shutdown(wait=False, cancel_futures=True)

        raise

    finally:
        save_repair_partial(results_by_index)

        if executor is not None and not cancelled:
            executor.shutdown(wait=True)

missing_after_repair_run = [
    idx for idx in repair_input_indices
    if idx not in results_by_index
]

if missing_after_repair_run:
    raise RuntimeError(
        f"{len(missing_after_repair_run)} repair items are still missing. "
        f"Re-run this cell to resume from: {REPAIR_PARTIAL_PATH}"
    )

repair_results = [
    results_by_index[idx]
    for idx in sorted(results_by_index)
]

total_time_sec = time.perf_counter() - overall_start

num_errors = sum(1 for x in repair_results if x.get("error") is not None)
num_usable_success = sum(
    1 for x in repair_results
    if is_usable_success(x, x["input_index"])
)

latencies = [
    x["latency_sec"]
    for x in repair_results
    if x.get("latency_sec") is not None
]

token_budget_counts = Counter(str(x.get("max_tokens_used")) for x in repair_results)
finish_reason_counts = Counter(str(x.get("finish_reason")) for x in repair_results)

repair_meta = {
    "dataset": DATASET_NAME,
    "input_path": str(GDRIVE_INPUT_PATH),
    "repair_index_path": str(REPAIR_INDEX_PATH),
    "output_path": str(REPAIR_OUTPUT_PATH),
    "partial_path": str(REPAIR_PARTIAL_PATH),
    "model": MODEL_NAME,
    "num_repair_items": len(repair_results),
    "num_usable_success": num_usable_success,
    "num_errors": num_errors,
    "repair_input_index_ranges_0_based": summarize_ranges([x["input_index"] for x in repair_results]),
    "repair_chunk_number_ranges_1_based": summarize_ranges([x["input_index"] + 1 for x in repair_results]),
    "max_workers": MAX_WORKERS,
    "save_every": SAVE_EVERY,
    "output_token_steps": REPAIR_OUTPUT_TOKEN_STEPS,
    "temperature": 0.0,
    "top_p": 1.0,
    "top_k": 20,
    "min_p": 0.0,
    "presence_penalty": 0.0,
    "repetition_penalty": 1.12,
    "token_budget_counts": dict(token_budget_counts),
    "finish_reason_counts": dict(finish_reason_counts),
    "total_time_sec": round(total_time_sec, 3),
    "avg_latency_sec": round(sum(latencies) / len(latencies), 3) if latencies else None,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
}

atomic_json_dump(repair_results, REPAIR_OUTPUT_PATH)
atomic_json_dump(repair_meta, REPAIR_META_PATH)
save_repair_partial(results_by_index)

print("Saved repair JSON:", REPAIR_OUTPUT_PATH)
print("Saved repair checkpoint JSON:", REPAIR_PARTIAL_PATH)
print("Saved repair meta JSON:", REPAIR_META_PATH)
print("Repair usable successes:", num_usable_success, "/", len(repair_results))
print("Repair errors:", num_errors)

Repair items to process: 16
Repair chunk numbers: ['1583', '1609', '1672', '1785', '2350', '3804', '4796', '5626', '6046', '6349', '7688', '8736', '8919', '10634', '10637', '10779']
Already completed usable repair items: 0
Pending repair items: 16


Repairing missing or failed KG items:   0%|          | 0/16 [00:00<?, ?it/s]

Saved repair JSON: /content/drive/MyDrive/final_project/idea_1/kg/2wikimultihopqa/2wikimultihopqa_kg_extractions_repair_missing_failed.json
Saved repair checkpoint JSON: /content/drive/MyDrive/final_project/idea_1/kg/2wikimultihopqa/2wikimultihopqa_kg_extractions_repair_missing_failed.partial.json
Saved repair meta JSON: /content/drive/MyDrive/final_project/idea_1/kg/2wikimultihopqa/2wikimultihopqa_kg_extractions_repair_missing_failed_meta.json
Repair usable successes: 14 / 16
Repair errors: 2


In [11]:
# 11) Final audit after adding repair file
import json
from collections import defaultdict, Counter
from datetime import datetime, timezone

FINAL_AUDIT_REPORT_PATH = KG_DIR / "2wikimultihopqa_kg_audit_report_after_repair.json"

combined_paths = final_paths.copy()

if REPAIR_OUTPUT_PATH.exists():
    combined_paths.append(REPAIR_OUTPUT_PATH)

final_records_by_index = defaultdict(list)
final_invalid_records = []
final_mismatched_records = []

for path in combined_paths:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, list):
        final_invalid_records.append({
            "file": str(path),
            "reason": "File content is not a list.",
        })
        continue

    for local_pos, item in enumerate(data):
        if not isinstance(item, dict):
            final_invalid_records.append({
                "file": str(path),
                "local_pos": local_pos,
                "reason": "Item is not a dict.",
            })
            continue

        idx = input_index_from_item(item)

        if not isinstance(idx, int) or idx < 0 or idx >= len(chunks):
            final_invalid_records.append({
                "file": str(path),
                "local_pos": local_pos,
                "input_index": idx,
                "chunk_id": item.get("chunk_id"),
                "reason": "Invalid input_index.",
            })
            continue

        expected_id = expected_chunk_id(idx)
        got_id = item.get("chunk_id")

        if got_id != expected_id:
            final_mismatched_records.append({
                "file": str(path),
                "local_pos": local_pos,
                "input_index": idx,
                "expected_chunk_id": expected_id,
                "got_chunk_id": got_id,
            })

        final_records_by_index[idx].append({
            "file": str(path),
            "filename": path.name,
            "local_pos": local_pos,
            "chunk_id": got_id,
            "error": item.get("error"),
            "usable_success": is_usable_success(item, idx),
            "failure_reasons": failure_reasons_hotpot_style(item, idx),
        })

final_covered_indices = set(final_records_by_index.keys())
final_missing_input_indices = sorted(set(range(len(chunks))) - final_covered_indices)

final_error_input_indices = sorted(
    idx for idx, rows in final_records_by_index.items()
    if any(row.get("error") is not None for row in rows)
)

final_unusable_input_indices = sorted(
    idx for idx, rows in final_records_by_index.items()
    if not any(row["usable_success"] for row in rows)
)

final_duplicate_input_indices = sorted(
    idx for idx, rows in final_records_by_index.items()
    if len(rows) > 1
)

final_audit_report = {
    "dataset": DATASET_NAME,
    "total_chunks": len(chunks),
    "num_files_checked": len(combined_paths),
    "files_checked": [str(p) for p in combined_paths],
    "num_covered_input_indices": len(final_covered_indices),
    "num_missing_input_indices": len(final_missing_input_indices),
    "num_error_input_indices": len(final_error_input_indices),
    "num_unusable_input_indices": len(final_unusable_input_indices),
    "num_duplicate_input_indices": len(final_duplicate_input_indices),
    "num_invalid_records": len(final_invalid_records),
    "num_mismatched_records": len(final_mismatched_records),
    "missing_input_index_ranges_0_based": summarize_ranges(final_missing_input_indices),
    "missing_chunk_number_ranges_1_based": summarize_ranges([i + 1 for i in final_missing_input_indices]),
    "error_input_index_ranges_0_based": summarize_ranges(final_error_input_indices),
    "error_chunk_number_ranges_1_based": summarize_ranges([i + 1 for i in final_error_input_indices]),
    "unusable_input_index_ranges_0_based": summarize_ranges(final_unusable_input_indices),
    "unusable_chunk_number_ranges_1_based": summarize_ranges([i + 1 for i in final_unusable_input_indices]),
    "duplicate_input_indices": final_duplicate_input_indices,
    "invalid_records": final_invalid_records,
    "mismatched_records": final_mismatched_records,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
}

atomic_json_dump(final_audit_report, FINAL_AUDIT_REPORT_PATH)

print("Final covered input indices:", len(final_covered_indices), "/", len(chunks))
print("Final missing input indices:", len(final_missing_input_indices))
print("Final missing chunk numbers:", summarize_ranges([i + 1 for i in final_missing_input_indices]))
print("Final error input indices:", len(final_error_input_indices))
print("Final error chunk numbers:", summarize_ranges([i + 1 for i in final_error_input_indices]))
print("Final unusable input indices:", len(final_unusable_input_indices))
print("Final unusable chunk numbers:", summarize_ranges([i + 1 for i in final_unusable_input_indices]))
print("Final duplicate input indices:", len(final_duplicate_input_indices))
print("Final mismatched records:", len(final_mismatched_records))
print("Saved final audit report:", FINAL_AUDIT_REPORT_PATH)

if final_missing_input_indices:
    raise RuntimeError("Some chunks are still missing. Check the final audit report.")

if final_unusable_input_indices:
    print("Warning: some chunks are still unusable. Safe retry will run in the next cell.")
else:
    print("All chunks are covered and at least one usable record exists per input_index.")

Final covered input indices: 12685 / 12685
Final missing input indices: 0
Final missing chunk numbers: []
Final error input indices: 16
Final error chunk numbers: ['1583', '1609', '1672', '1785', '2350', '3804', '4796', '5626', '6046', '6349', '7688', '8736', '8919', '10634', '10637', '10779']
Final unusable input indices: 2
Final unusable chunk numbers: ['6046', '10637']
Final duplicate input indices: 16
Final mismatched records: 0
Saved final audit report: /content/drive/MyDrive/final_project/idea_1/kg/2wikimultihopqa/2wikimultihopqa_kg_audit_report_after_repair.json


In [12]:
# 12) Safe retry for remaining truly unusable records
import copy
import json
import os
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
from collections import Counter
from tqdm.auto import tqdm

SAFE_REPAIR_OUTPUT_PATH = KG_DIR / "2wikimultihopqa_kg_extractions_repair_safe_retry.json"
SAFE_REPAIR_PARTIAL_PATH = KG_DIR / "2wikimultihopqa_kg_extractions_repair_safe_retry.partial.json"
SAFE_REPAIR_META_PATH = KG_DIR / "2wikimultihopqa_kg_extractions_repair_safe_retry_meta.json"

SAFE_RETRY_INDICES = final_unusable_input_indices

SAFE_MAX_WORKERS = 4
SAFE_SAVE_EVERY = 5
SAFE_OUTPUT_TOKEN_STEPS = [2048, 3072, 4096, 6144]

print("Safe retry items:", len(SAFE_RETRY_INDICES))
print("Safe retry chunk numbers:", summarize_ranges([i + 1 for i in SAFE_RETRY_INDICES]))

SAFE_KG_SCHEMA = copy.deepcopy(KG_SCHEMA)

SAFE_KG_SCHEMA["properties"]["entities"]["maxItems"] = 20
SAFE_KG_SCHEMA["properties"]["entities"]["items"]["maxLength"] = 140

SAFE_KG_SCHEMA["properties"]["relations"]["maxItems"] = 24
SAFE_KG_SCHEMA["properties"]["relations"]["items"]["properties"]["head"]["maxLength"] = 140
SAFE_KG_SCHEMA["properties"]["relations"]["items"]["properties"]["tail"]["maxLength"] = 140
SAFE_KG_SCHEMA["properties"]["relations"]["items"]["properties"]["relation"]["maxLength"] = 280

SAFE_KG_SCHEMA["properties"]["facts"]["maxItems"] = 20
SAFE_KG_SCHEMA["properties"]["facts"]["items"]["properties"]["entity"]["maxLength"] = 140
SAFE_KG_SCHEMA["properties"]["facts"]["items"]["properties"]["info"]["maxLength"] = 280

def build_safe_prompt(chunk):
    # Short prompt to avoid pathological looping.
    return f"""Extract a compact knowledge graph from the Wikipedia chunk.

Return JSON only.
Do not explain.
Do not write analysis.
Do not use external knowledge.
Use only facts explicitly stated in the chunk.
Prefer precision over recall.

Limits:
- entities: at most 20 short strings
- relations: at most 24 objects
- facts: at most 20 objects

Required JSON keys:
entities, relations, facts

Chunk id:
{chunk["Chunk_id"]}

Title:
{chunk["Title"]}

Paragraph ids:
{json.dumps(chunk["Paragraph_id"], ensure_ascii=False)}

Text:
{chunk["Text"]}
"""

def call_llm_safe_once(chunk, max_tokens):
    # Compact retry with deterministic decoding.
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": "Return only compact valid JSON. No explanations."
            },
            {
                "role": "user",
                "content": build_safe_prompt(chunk)
            },
        ],
        max_tokens=max_tokens,
        temperature=0.0,
        top_p=1.0,
        presence_penalty=0.0,
        seed=202,
        extra_body={
            "top_k": 20,
            "min_p": 0.0,
            "repetition_penalty": 1.18,
            "chat_template_kwargs": {
                "enable_thinking": False
            },
            "structured_outputs": {
                "json": SAFE_KG_SCHEMA
            },
        },
    )

    choice = response.choices[0]

    return {
        "content": choice.message.content,
        "finish_reason": choice.finish_reason,
        "usage": usage_to_dict(response.usage),
        "max_tokens": max_tokens,
    }

def extract_one_safe(chunk, max_retries_per_budget=2):
    # Safer extraction for items that still failed after normal repair.
    start = time.perf_counter()
    attempts = []
    last_error = None
    raw = None
    last_finish_reason = None
    last_usage = None
    last_max_tokens = None

    for max_tokens in SAFE_OUTPUT_TOKEN_STEPS:
        for retry_id in range(max_retries_per_budget + 1):
            try:
                response_data = call_llm_safe_once(chunk, max_tokens=max_tokens)

                raw = response_data["content"]
                finish_reason = response_data["finish_reason"]
                usage = response_data["usage"]

                last_finish_reason = finish_reason
                last_usage = usage
                last_max_tokens = max_tokens

                attempts.append({
                    "max_tokens": max_tokens,
                    "retry_id": retry_id,
                    "finish_reason": finish_reason,
                    "usage": usage,
                })

                if finish_reason == "length":
                    last_error = f"finish_reason=length at max_tokens={max_tokens}"
                    break

                model_output = json.loads(raw)
                validate_repair_payload_hotpot_style(model_output)

                return {
                    "chunk_id": chunk["Chunk_id"],
                    "title": chunk["Title"],
                    "paragraph_id": chunk["Paragraph_id"],
                    "token_count": chunk.get("Token_count"),
                    "entities": model_output.get("entities"),
                    "relations": model_output.get("relations"),
                    "facts": model_output.get("facts"),
                    "error": None,
                    "latency_sec": round(time.perf_counter() - start, 3),
                    "finish_reason": finish_reason,
                    "max_tokens_used": max_tokens,
                    "usage": usage,
                    "attempts": attempts,
                    "repair_run": True,
                    "repair_pass": "safe_retry",
                }

            except Exception as e:
                last_error = repr(e)
                time.sleep(1.0 * (retry_id + 1))

    failed_item = {
        "chunk_id": chunk["Chunk_id"],
        "title": chunk["Title"],
        "paragraph_id": chunk["Paragraph_id"],
        "token_count": chunk.get("Token_count"),
        "entities": None,
        "relations": None,
        "facts": None,
        "error": last_error,
        "latency_sec": round(time.perf_counter() - start, 3),
        "finish_reason": last_finish_reason,
        "max_tokens_used": last_max_tokens,
        "usage": last_usage,
        "attempts": attempts,
        "raw_response": raw,
        "repair_run": True,
        "repair_pass": "safe_retry",
    }

    return failed_item

def load_existing_safe_items():
    # Prefer final safe retry output, then checkpoint.
    for path in [SAFE_REPAIR_OUTPUT_PATH, SAFE_REPAIR_PARTIAL_PATH]:
        if path.exists():
            try:
                return load_json_list_if_exists(path)
            except Exception:
                pass

    return []

def save_safe_partial(results_by_index):
    # Save completed safe retry items.
    partial_results = [
        results_by_index[idx]
        for idx in sorted(results_by_index)
    ]
    atomic_json_dump(partial_results, SAFE_REPAIR_PARTIAL_PATH)

safe_index_to_local = {
    idx: local_pos
    for local_pos, idx in enumerate(SAFE_RETRY_INDICES)
}

safe_results_by_index = {}

for item in load_existing_safe_items():
    idx = input_index_from_item(item)

    if idx not in safe_index_to_local:
        continue

    if not is_usable_success(item, idx):
        continue

    item["input_index"] = idx
    item["safe_retry_local_index"] = safe_index_to_local[idx]
    safe_results_by_index[idx] = item

pending_safe_jobs = [
    (idx, chunks[idx])
    for idx in SAFE_RETRY_INDICES
    if idx not in safe_results_by_index
]

print("Already successful safe retry items:", len(safe_results_by_index))
print("Pending safe retry items:", len(pending_safe_jobs))

safe_start = time.perf_counter()
completed_new = 0

if pending_safe_jobs:
    with ThreadPoolExecutor(max_workers=SAFE_MAX_WORKERS) as executor:
        futures = {
            executor.submit(extract_one_safe, chunk): (idx, chunk)
            for idx, chunk in pending_safe_jobs
        }

        for future in tqdm(
            as_completed(futures),
            total=len(futures),
            desc="Safe retry remaining unusable items"
        ):
            idx, chunk = futures[future]

            try:
                item = future.result()
            except Exception as e:
                item = {
                    "chunk_id": chunk["Chunk_id"],
                    "title": chunk["Title"],
                    "paragraph_id": chunk["Paragraph_id"],
                    "token_count": chunk.get("Token_count"),
                    "entities": None,
                    "relations": None,
                    "facts": None,
                    "error": repr(e),
                    "latency_sec": None,
                    "finish_reason": None,
                    "max_tokens_used": None,
                    "usage": None,
                    "attempts": [],
                    "repair_run": True,
                    "repair_pass": "safe_retry",
                }

            item["input_index"] = idx
            item["safe_retry_local_index"] = safe_index_to_local[idx]
            item["repair_run"] = True
            item["repair_pass"] = "safe_retry"

            safe_results_by_index[idx] = item
            completed_new += 1

            if completed_new % SAFE_SAVE_EVERY == 0:
                save_safe_partial(safe_results_by_index)

    save_safe_partial(safe_results_by_index)

safe_retry_results = [
    safe_results_by_index[idx]
    for idx in sorted(safe_results_by_index)
]

safe_retry_errors = [
    x["input_index"]
    for x in safe_retry_results
    if not is_usable_success(x, x["input_index"])
]

safe_meta = {
    "dataset": DATASET_NAME,
    "model": MODEL_NAME,
    "num_safe_retry_items": len(safe_retry_results),
    "num_safe_retry_errors": len(safe_retry_errors),
    "safe_retry_error_chunk_numbers": summarize_ranges([i + 1 for i in safe_retry_errors]),
    "safe_schema": "compact_with_string_maxLength",
    "output_token_steps": SAFE_OUTPUT_TOKEN_STEPS,
    "temperature": 0.0,
    "top_p": 1.0,
    "repetition_penalty": 1.18,
    "max_workers": SAFE_MAX_WORKERS,
    "total_time_sec": round(time.perf_counter() - safe_start, 3),
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
}

atomic_json_dump(safe_retry_results, SAFE_REPAIR_OUTPUT_PATH)
atomic_json_dump(safe_meta, SAFE_REPAIR_META_PATH)
save_safe_partial(safe_results_by_index)

print("Saved safe retry JSON:", SAFE_REPAIR_OUTPUT_PATH)
print("Saved safe retry checkpoint JSON:", SAFE_REPAIR_PARTIAL_PATH)
print("Saved safe retry meta JSON:", SAFE_REPAIR_META_PATH)
print("Safe retry remaining errors:", len(safe_retry_errors))
print("Safe retry remaining error chunk numbers:", summarize_ranges([i + 1 for i in safe_retry_errors]))

Safe retry items: 2
Safe retry chunk numbers: ['6046', '10637']
Already successful safe retry items: 0
Pending safe retry items: 2


Safe retry remaining unusable items:   0%|          | 0/2 [00:00<?, ?it/s]

Saved safe retry JSON: /content/drive/MyDrive/final_project/idea_1/kg/2wikimultihopqa/2wikimultihopqa_kg_extractions_repair_safe_retry.json
Saved safe retry checkpoint JSON: /content/drive/MyDrive/final_project/idea_1/kg/2wikimultihopqa/2wikimultihopqa_kg_extractions_repair_safe_retry.partial.json
Saved safe retry meta JSON: /content/drive/MyDrive/final_project/idea_1/kg/2wikimultihopqa/2wikimultihopqa_kg_extractions_repair_safe_retry_meta.json
Safe retry remaining errors: 1
Safe retry remaining error chunk numbers: ['10637']


In [13]:
# 13) Build one clean merged final file
import copy
from collections import defaultdict, Counter
from datetime import datetime, timezone

CLEAN_OUTPUT_PATH = KG_DIR / f"2wikimultihopqa_kg_extractions_all_00000001_to_{len(chunks):08d}.clean.json"
CLEAN_AUDIT_PATH = KG_DIR / f"2wikimultihopqa_kg_extractions_all_00000001_to_{len(chunks):08d}.clean_audit.json"

def get_source_specs_for_clean_merge():
    # Higher priority wins if multiple records exist for the same input_index.
    specs = []

    original_paths = sorted(
        p for p in KG_DIR.glob("2wikimultihopqa_kg_extractions_*.json")
        if FINAL_OUTPUT_RE.match(p.name)
    )

    for path in original_paths:
        specs.append({
            "path": path,
            "source_kind": "original_batch",
            "source_priority": 10,
        })

    if REPAIR_OUTPUT_PATH.exists():
        specs.append({
            "path": REPAIR_OUTPUT_PATH,
            "source_kind": "repair_missing_failed",
            "source_priority": 20,
        })
    elif REPAIR_PARTIAL_PATH.exists():
        specs.append({
            "path": REPAIR_PARTIAL_PATH,
            "source_kind": "repair_missing_failed_partial",
            "source_priority": 20,
        })

    if SAFE_REPAIR_OUTPUT_PATH.exists():
        specs.append({
            "path": SAFE_REPAIR_OUTPUT_PATH,
            "source_kind": "safe_retry",
            "source_priority": 30,
        })
    elif SAFE_REPAIR_PARTIAL_PATH.exists():
        specs.append({
            "path": SAFE_REPAIR_PARTIAL_PATH,
            "source_kind": "safe_retry_partial",
            "source_priority": 30,
        })

    return specs

def collect_clean_candidates():
    # Collect candidate records from original and repair files.
    candidates = defaultdict(list)
    source_summaries = []

    for spec in get_source_specs_for_clean_merge():
        path = spec["path"]
        source_kind = spec["source_kind"]
        source_priority = spec["source_priority"]

        if not path.exists():
            continue

        data = load_json_list_if_exists(path)
        bad_items = 0

        for local_pos, item in enumerate(data):
            if not isinstance(item, dict):
                bad_items += 1
                continue

            idx = input_index_from_item(item)

            if not isinstance(idx, int) or idx < 0 or idx >= len(chunks):
                bad_items += 1
                continue

            candidates[idx].append({
                "item": item,
                "path": str(path),
                "filename": path.name,
                "local_pos": local_pos,
                "source_kind": source_kind,
                "source_priority": source_priority,
                "usable_success": is_usable_success(item, idx),
                "failure_reasons": failure_reasons_hotpot_style(item, idx),
            })

        source_summaries.append({
            "path": str(path),
            "filename": path.name,
            "source_kind": source_kind,
            "source_priority": source_priority,
            "num_items": len(data),
            "bad_items": bad_items,
        })

    return candidates, source_summaries

def choose_best_candidate(rows):
    # Prefer usable records, then newer repair source, then non-length records.
    def score(row):
        item = row["item"]

        usable_score = 1 if row["usable_success"] else 0
        source_priority = row["source_priority"]
        finish_not_length = 1 if item.get("finish_reason") != "length" else 0
        has_any_output = int(
            isinstance(item.get("entities"), list)
            or isinstance(item.get("relations"), list)
            or isinstance(item.get("facts"), list)
        )

        return (
            usable_score,
            source_priority,
            finish_not_length,
            has_any_output,
            -row["local_pos"],
        )

    return max(rows, key=score)

def make_missing_placeholder(idx):
    # Create a placeholder if a chunk is still completely missing.
    chunk = chunks[idx]

    return {
        "chunk_id": chunk["Chunk_id"],
        "title": chunk["Title"],
        "paragraph_id": chunk["Paragraph_id"],
        "token_count": chunk.get("Token_count"),
        "entities": None,
        "relations": None,
        "facts": None,
        "error": "missing_after_all_repairs",
        "latency_sec": None,
        "finish_reason": None,
        "max_tokens_used": None,
        "usage": None,
        "attempts": [],
        "input_index": idx,
        "final_clean_source_file": None,
        "final_clean_source_kind": "missing_placeholder",
    }

candidates, source_summaries = collect_clean_candidates()

clean_items = []
clean_items_by_index = {}

missing_indices = []
true_unusable_indices = []
duplicate_source_indices = []
chosen_sources = Counter()
failure_reasons_by_input_index = {}

for idx in range(len(chunks)):
    rows = candidates.get(idx, [])

    if not rows:
        item = make_missing_placeholder(idx)
        clean_items.append(item)
        clean_items_by_index[idx] = item
        missing_indices.append(idx)
        true_unusable_indices.append(idx)
        chosen_sources["missing_placeholder"] += 1
        failure_reasons_by_input_index[str(idx)] = ["missing_after_all_repairs"]
        continue

    if len(rows) > 1:
        duplicate_source_indices.append(idx)

    best = choose_best_candidate(rows)
    item = copy.deepcopy(best["item"])

    item["input_index"] = idx
    item["final_clean_source_file"] = best["filename"]
    item["final_clean_source_kind"] = best["source_kind"]

    clean_items.append(item)
    clean_items_by_index[idx] = item
    chosen_sources[best["source_kind"]] += 1

    if not is_usable_success(item, idx):
        true_unusable_indices.append(idx)
        failure_reasons_by_input_index[str(idx)] = failure_reasons_hotpot_style(item, idx)

clean_audit = {
    "dataset": DATASET_NAME,
    "input_path": str(GDRIVE_INPUT_PATH),
    "kg_dir": str(KG_DIR),
    "clean_output_path": str(CLEAN_OUTPUT_PATH),
    "total_chunks": len(chunks),
    "num_clean_items": len(clean_items),
    "num_missing_indices": len(missing_indices),
    "num_true_unusable_indices": len(true_unusable_indices),
    "num_duplicate_source_indices_ignored": len(duplicate_source_indices),
    "missing_input_index_ranges_0_based": summarize_ranges(missing_indices),
    "missing_chunk_number_ranges_1_based": summarize_ranges([i + 1 for i in missing_indices]),
    "true_unusable_input_index_ranges_0_based": summarize_ranges(true_unusable_indices),
    "true_unusable_chunk_number_ranges_1_based": summarize_ranges([i + 1 for i in true_unusable_indices]),
    "duplicate_input_index_ranges_0_based": summarize_ranges(duplicate_source_indices),
    "chosen_sources": dict(chosen_sources),
    "source_summaries": source_summaries,
    "failure_reasons_by_input_index": failure_reasons_by_input_index,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
}

atomic_json_dump(clean_items, CLEAN_OUTPUT_PATH)
atomic_json_dump(clean_audit, CLEAN_AUDIT_PATH)

print("Final clean output")
print("Saved clean JSON:", CLEAN_OUTPUT_PATH)
print("Saved clean audit:", CLEAN_AUDIT_PATH)
print("Clean items:", len(clean_items), "/", len(chunks))
print("Missing:", len(missing_indices))
print("True unusable:", len(true_unusable_indices))
print("True unusable chunk numbers:", summarize_ranges([i + 1 for i in true_unusable_indices]))
print("Duplicate source indices ignored:", len(duplicate_source_indices))
print("Chosen sources:", dict(chosen_sources))

if missing_indices:
    print("Warning: some chunks are still missing and were stored as placeholders.")

if true_unusable_indices:
    print("Warning: some chunks still failed after all repair passes.")
else:
    print("All chunks are covered with one clean usable record per input_index.")

Final clean output
Saved clean JSON: /content/drive/MyDrive/final_project/idea_1/kg/2wikimultihopqa/2wikimultihopqa_kg_extractions_all_00000001_to_00012685.clean.json
Saved clean audit: /content/drive/MyDrive/final_project/idea_1/kg/2wikimultihopqa/2wikimultihopqa_kg_extractions_all_00000001_to_00012685.clean_audit.json
Clean items: 12685 / 12685
Missing: 0
True unusable: 1
True unusable chunk numbers: ['10637']
Duplicate source indices ignored: 16
Chosen sources: {'original_batch': 12669, 'repair_missing_failed': 14, 'safe_retry': 2}


In [14]:
# 14) Save and print unresolved chunks after all repair passes
import json

UNRESOLVED_CHUNKS_PATH = KG_DIR / "2wikimultihopqa_kg_unresolved_chunks_after_all_repairs.json"

unresolved_details = []

for idx in true_unusable_indices:
    chunk = chunks[idx]
    best_item = clean_items_by_index[idx]

    unresolved_details.append({
        "input_index_0_based": idx,
        "chunk_number_1_based": idx + 1,
        "chunk_id": chunk["Chunk_id"],
        "title": chunk.get("Title"),
        "paragraph_id": chunk.get("Paragraph_id"),
        "token_count": chunk.get("Token_count"),
        "failure_reasons": failure_reasons_hotpot_style(best_item, idx),
        "best_error": best_item.get("error"),
        "best_finish_reason": best_item.get("finish_reason"),
        "best_source_file": best_item.get("final_clean_source_file"),
        "best_source_kind": best_item.get("final_clean_source_kind"),
        "text": chunk.get("Text"),
    })

atomic_json_dump(unresolved_details, UNRESOLVED_CHUNKS_PATH)

print("Saved unresolved chunks JSON:", UNRESOLVED_CHUNKS_PATH)
print("Number of unresolved chunks:", len(unresolved_details))
print("Unresolved chunk numbers:", summarize_ranges([x["chunk_number_1_based"] for x in unresolved_details]))

for item in unresolved_details:
    print("\n" + "=" * 120)
    print("input_index_0_based:", item["input_index_0_based"])
    print("chunk_number_1_based:", item["chunk_number_1_based"])
    print("chunk_id:", item["chunk_id"])
    print("title:", item["title"])
    print("paragraph_id:", item["paragraph_id"])
    print("token_count:", item["token_count"])
    print("failure_reasons:", item["failure_reasons"])
    print("best_error:", item["best_error"])
    print("best_finish_reason:", item["best_finish_reason"])
    print("best_source_file:", item["best_source_file"])
    print("best_source_kind:", item["best_source_kind"])
    print("-" * 120)
    print(item["text"])

Saved unresolved chunks JSON: /content/drive/MyDrive/final_project/idea_1/kg/2wikimultihopqa/2wikimultihopqa_kg_unresolved_chunks_after_all_repairs.json
Number of unresolved chunks: 1
Unresolved chunk numbers: ['10637']

input_index_0_based: 10636
chunk_number_1_based: 10637
chunk_id: 2wikimultihopqa_chunk_00010637
title: Youssef Chahine
paragraph_id: [1]
token_count: 124
failure_reasons: ['entities_not_list', 'error_not_none', 'facts_not_list', 'finish_reason_length', 'relations_not_list']
best_error: finish_reason=length at max_tokens=6144
best_finish_reason: length
best_source_file: 2wikimultihopqa_kg_extractions_repair_safe_retry.json
best_source_kind: safe_retry
------------------------------------------------------------------------------------------------------------------------
Youssef Chahine (; 25 January 1926 – 27 July 2008) was an Egyptian film director. He was active in the Egyptian film industry from 1950 until his death. A winner of the Cannes 50th Anniversary Award (for